---
title: "Flujo de trabajo - Preparación de datos de ecosistemas del país"
subtitle: "Taller de evaluaciones RLE"
author:
  - name: "Tyler Erickson"
date: "2026-07-23"
execute:
  enabled: false
  freeze: false
  cache: false
lang: es
format:
  revealjs:
    theme: night
    css: styles.css
    slide-number: true
    navigation-mode: vertical
---


## Configuración de la presentación {visibility="hidden"}

```{=html}
<style>
[data-class-output="small-output"] .cell-output table { font-size: 0.4em; }
</style>
```

## Resumen

Esta presentación demuestra cómo preparar datos de mapas de ecosistemas de un país para un análisis eficiente.

Utiliza a Colombia como ejemplo.

## Fuente de los datos

El mapa de ecosistemas de 2024 está publicado en el [sitio web de IDEAM](https://www.ideam.gov.co/), que contiene una [página con el listado de conjuntos de datos relacionados con ecosistemas](https://www.ideam.gov.co/ecosistemas).

## Portal de Ecosistemas de IDEAM

:::: {.columns}

::: {.column width="60%"}
La página de ecosistemas de [IDEAM](https://www.ideam.gov.co/) alberga los conjuntos de datos nacionales de ecosistemas de Colombia.

Desde aquí puede explorar y descargar el Mapa de Ecosistemas Continentales, Costeros y Marinos (MEC) utilizado en este flujo de trabajo.

Los datos de ecosistemas se distribuyen como un archivo zip de **~2.3 GB**, que contiene los datos tanto en formato Shapefile como en Geodatabase. 

Haremos copias en formatos optimizados para la nube, para permitir flujos de trabajo adicionales.
:::

::: {.column width="40%"}
[![Portal de ecosistemas de IDEAM](images/ideam_ecosystem_webpage.png){height="500"}](https://www.ideam.gov.co/ecosistemas)
:::

::::

## Procesamiento Local {.scrollable}

Haz clic en [Descargar el shapefile del mapa de ecosistemas, continentales, costeros y marinos de Colombia (MEC)" 1:100,000 2024](https://e436.short.gy/MEcosis2024) que redirige a un enlace de Sharepoint protegido por inicio de sesión. Descarga el archivo manualmente a tu computador local.

Una vez se descargue, actualiza la siguiente celda con la ruta real.

In [16]:
from pathlib import Path

zip_path = Path("~/Downloads/Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024.zip").expanduser()

Inspecciona el archivo para determinar el formato.

In [17]:
#| echo: true
# Inspect the file to determine the format.
import zipfile

# Peek inside the zip without extracting it.
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()

print(f"{len(names)} entries in the zip; first few:")
for name in names[:5]:
    print("   ", name)

# Identify the geospatial data sources present (the zip may hold more than one).
gdbs = sorted({n[: n.index(".gdb/") + 4] for n in names if ".gdb/" in n})
shps = sorted(n for n in names if n.lower().endswith(".shp"))

print()
print("File Geodatabases:", gdbs or "none")
print("Shapefiles:       ", shps or "none")

67 entries in the zip; first few:
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/a00000001.freelist
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/a00000001.gdbindexes
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/a00000001.gdbtable

File Geodatabases: ['Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb']
Shapefiles:        ['Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/SHAPE/e_eccmc_100K_2024.shp']


El archivo zip contiene la misma capa de ecosistemas en dos formatos: una [Esri File Geodatabase](https://doc.esri.com/en/arcgis-pro/latest/help/data/geodatabases/manage-file-gdb/file-geodatabases.html) y un [Shapefile](https://doc.arcgis.com/en/arcgis-online/reference/shapefiles.htm). Nosotros leeremos la geodatabase.

In [ ]:
#| echo: true
# Build the GDAL /vsizip/ path pointing at the .gdb directory inside the zip.
gdb_source = (
    f"/vsizip/{zip_path}"
    "/Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024"
    "/ECOSISTEMAS_MEC_122024.gdb"
)

Lee la geodatabase en un GeoDataFrame. (Esto puede tardar hasta 60 segundos.)

In [ ]:
#| echo: true
import geopandas as gpd

gdf = gpd.read_file(gdb_source)

Mostrar los datos de MEC.

In [ ]:
#| echo: true
gdf

,tipo_ecos,gra_trans,gran_bioma,bioma_preliminar,bioma_IAvH,ecos_sintesis,ecos_general,u_sintesis,amb_acuatico,subsistema,...,no_anfibio,no_aves,no_magnoliops,no_mamiferos,no_reptiles,ruleid,override,Shape_Length,Shape_Area,geometry
0,Acuatico,Natural,Pedobioma del Zonobioma Humedo Tropical,Hidrobioma,Hidrobioma Alto Caquetá,Rio,Rio de Aguas Blancas,Rio de Aguas Blancas de la zona hidrográfica C...,Lotico,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,66,None,2.477929,0.003581,"MULTIPOLYGON (((-75.91369 0.9763, -75.91345 0...."
1,Acuatico,Transformado,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Transicional Transformado,Transicional Transformado,Transicional Transformado en áreas que predomi...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,75,None,0.028903,0.000023,"MULTIPOLYGON (((-75.52794 0.85907, -75.52701 0..."
2,Acuatico,Natural,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Bosque,Bosque Inundable Basal,Bosque Inundable Basal en áreas que predomina ...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,29,None,0.112034,0.000376,"MULTIPOLYGON (((-75.51411 0.79744, -75.514 0.7..."
3,Acuatico,Transformado,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Transicional Transformado,Transicional Transformado,Transicional Transformado en áreas que predomi...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,75,None,0.031162,0.000027,"MULTIPOLYGON (((-75.52821 0.87225, -75.5277 0...."
4,Acuatico,Transformado,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Transicional Transformado,Transicional Transformado,Transicional Transformado en áreas que predomi...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,75,None,0.038580,0.000024,"MULTIPOLYGON (((-75.52416 0.79585, -75.52418 0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460345,Terrestre,Natural,Pedobioma del Zonobioma Humedo Tropical,Litobioma,Litobioma Serranía del Naquén,Complejos Rocosos,Complejos Rocosos de Serranias,Complejos Rocosos de Serranias en áreas que pr...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,40,None,0.033453,0.000040,"MULTIPOLYGON (((-68.21295 1.97073, -68.2133 1...."
460346,Terrestre,Natural,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical Serranía del Naquén,Bosque,Bosque Basal Humedo,Bosque Basal Humedo en áreas que predomina Bos...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,24,None,0.020580,0.000025,"MULTIPOLYGON (((-67.49606 2.17007, -67.4946 2...."
460347,Terrestre,Natural,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical Serranía del Naquén,Bosque,Bosque Basal Humedo,Bosque Basal Humedo en áreas que predomina Bos...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,24,None,0.033420,0.000028,"MULTIPOLYGON (((-67.48783 2.17801, -67.48786 2..."
460348,Terrestre,Natural,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical Serranía del Naquén,Bosque,Bosque Basal Humedo,Bosque Basal Humedo en áreas que predomina Bos...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,24,None,0.020843,0.000025,"MULTIPOLYGON (((-67.34087 1.98014, -67.34116 1..."


In [ ]:
#| echo: true
# Summarize the number of unique values of ecosystem columns.
print(f'{gdf["ecos_sintesis"].nunique() = }')
print(f'{gdf["ecos_general"].nunique() = }')
print(f'{gdf["u_sintesis"].nunique() = }')

gdf["ecos_sintesis"].nunique() = 28
gdf["ecos_general"].nunique() = 87
gdf["u_sintesis"].nunique() = 14122


In [ ]:
#| echo: true
# Sort by ecosystem so related records are grouped together.
gdf = gdf.sort_values("ecos_general", ignore_index=True)

# Write the MEC data to a local parquet file.
gdf.to_parquet("data/ECOSISTEMAS_MEC_122024.parquet", compression="zstd")

Aquí tienes un pequeño dataset de 5 registros para agilizar el desarrollo/las pruebas.

```json
[
  {
    "id": 1,
    "name": "Sample Alpha",
    "category": "A",
    "value": 12.5,
    "active": true
  },
  {
    "id": 2,
    "name": "Sample Beta",
    "category": "B",
    "value": 7.3,
    "active": false
  },
  {
    "id": 3,
    "name": "Sample Gamma",
    "category": "A",
    "value": 21.0,
    "active": true
  },
  {
    "id": 4,
    "name": "Sample Delta",
    "category": "C",
    "value": 4.8,
    "active": true
  },
  {
    "id": 5,
    "name": "Sample Epsilon",
    "category": "B",
    "value": 15.2,
    "active": false
  }
]
```

Si me indicas el esquema real que necesitas (nombres de campos, tipos de datos, formato deseado como CSV/JSON/GeoJSON/Parquet, o si debe incluir geometría/coordenadas), puedo ajustar este dataset de ejemplo para que coincida exactamente con tu caso de uso.

In [ ]:
gdf.head().to_parquet("data/ECOSISTEMAS_MEC_122024_5records.parquet", compression="zstd")

## Subconjunto del área de Bogotá

Crea un subconjunto de ecorregiones alrededor de Bogotá que será más fácil de manejar.

In [15]:
from shapely.geometry import Polygon

# Area of interest around Bogota (from an Earth Engine polygon, in EPSG:4326 lon/lat).
bogota_aoi = Polygon(
    [
        [-74.27360864858895, 4.898092789377364],
        [-74.27360864858895, 4.29853018346521],
        [-73.3809694884327, 4.29853018346521],
        [-73.3809694884327, 4.898092789377364],
    ]
)

# Reproject the AOI to match the ecosystem data's CRS before the spatial query.
bogota_aoi = gpd.GeoSeries([bogota_aoi], crs="EPSG:4326").to_crs(gdf.crs).iloc[0]

# Subset to the features that intersect the AOI.
gdf_bogata = gdf[gdf.intersects(bogota_aoi)]
gdf_bogata.shape

gdf_bogata.to_parquet("data/ECOSISTEMAS_MEC_122024_bogota_area.parquet", compression="zstd")

### Publicación en almacenamiento de objetos en la nube

La tabla de ecosistemas se convirtió a un formato nativo de la nube (parquet) y se publicó en Source Cooperative:

Página del producto: https://source.coop/tyler/colombia-ecosystems-map/ecosistemas/e_eccmc_100K_2024.parquet
URL de los datos: https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/e_eccmc_100K_2024.parquet

In [ ]:
# data_url = 'https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024.parquet'
# data_url = 'https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024_5records.parquet'
data_url = 'https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024_bogata_area.parquet'

## Inspeccionar el conjunto de datos publicado

Lee directamente desde el GeoParquet alojado en la nube y resume el tipo de geometría y el número de polígonos.

In [29]:
import geopandas as gpd
import fsspec

data = gpd.read_parquet(fsspec.open(data_url).open())

FileNotFoundError: https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024_bogata.parquet

In [9]:
data.shape

(460350, 50)

In [10]:
data.geometry.crs

<Geographic 2D CRS: EPSG:4686>
Name: MAGNA-SIRGAS
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: Colombia - onshore and offshore. Includes San Andres y Providencia, Malpelo Islands, Roncador Bank, Serrana Bank and Serranilla Bank.
- bounds: (-84.77, -4.23, -66.87, 15.51)
Datum: Marco Geocentrico Nacional de Referencia
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [11]:
print(data.geometry.geom_type.value_counts())                       # Polygon vs MultiPolygon


MultiPolygon    460350
Name: count, dtype: int64


In [12]:
data.columns

Index(['tipo_ecos', 'gra_trans', 'gran_bioma', 'bioma_preliminar',
       'bioma_IAvH', 'ecos_sintesis', 'ecos_general', 'u_sintesis',
       'amb_acuatico', 'subsistema', 'nom_zh', 'origen', 'tipo_agua', 'clima',
       'paisaje', 'relieve', 'suelos', 'amb_edafogenetico',
       'desc_amb_edafogenetico', 'cob', 'sustrato', 'zona', 'temperatura',
       'salinidad', 'provincia', 'eco_region', 'eco_zona', 'orig_marino',
       'config_biotica', 'clasif_biotica', 'subclasif_biotica',
       'grupo_biotico', 'sectores', 'area_ha', 'u_biotica', 'anfibios', 'aves',
       'magnoliops', 'mamiferos', 'reptiles', 'no_anfibio', 'no_aves',
       'no_magnoliops', 'no_mamiferos', 'no_reptiles', 'ruleid', 'override',
       'Shape_Length', 'Shape_Area', 'geometry'],
      dtype='str')

In [13]:
from itables import show

show(data["gran_bioma"].value_counts().to_frame("count"))

Loading ITables v2.8.1 from the internet... (need help?)


In [14]:
show(data["bioma_preliminar"].value_counts().to_frame("count"))

Loading ITables v2.8.1 from the internet... (need help?)


In [15]:
show(data["ecos_sintesis"].value_counts().to_frame("count"))

Loading ITables v2.8.1 from the internet... (need help?)


In [16]:
show(data["ecos_general"].value_counts().to_frame("count"))

Loading ITables v2.8.1 from the internet... (need help?)


In [17]:
show(data["bioma_IAvH"].value_counts().to_frame("count"))

Loading ITables v2.8.1 from the internet... (need help?)


In [18]:
show(data["u_sintesis"].value_counts().to_frame("count"))


Loading ITables v2.8.1 from the internet... (need help?)


In [19]:
gdf.head()['ecos_general']

0         Rio de Aguas Blancas
1    Transicional Transformado
2       Bosque Inundable Basal
3    Transicional Transformado
4    Transicional Transformado
Name: ecos_general, dtype: str